# QIFE Experiment on MNIST

This notebook implements Quantum Inspired Feature Extraction (QIFE) for MNIST image classification.

Recommended use:
1. Open this notebook in Google Colab from GitHub.
2. Select **Runtime → Disconnect and delete runtime**.
3. Select **Runtime → Run all**.
4. Verify that the dataset is downloaded automatically and the metrics are produced without requiring Google Drive files.


In [ ]:
# ===== CELL 1 : Install dependencies (if not installed) =====
!pip install torch pennylane tensorflow scikit-image medmnist

import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import pennylane as qml
from skimage.transform import resize
from torchvision import transforms
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import (precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score)
import time
from collections import defaultdict
import json
from pennylane import math
from torchvision import datasets

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
# ===== CELL 2 : Device & patch extractor =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def extract_patches(image, patch_size=3, stride=2):
    patches = image.unfold(2, patch_size, stride).unfold(3, patch_size, stride)
    patches = patches.contiguous().view(image.shape[0], 100, patch_size * patch_size)
    return patches.to(device)

In [ ]:
def load_mnist():
    """
    MNIST (10 kelas). Train subset: 1200/kelas, Test subset: 200/kelas.
    """
    ROOT = os.path.join(os.getcwd(), "torchvision_data")
    os.makedirs(ROOT, exist_ok=True)

    transform = transforms.Compose([
        transforms.Resize((22, 22)),
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])

    train_set = datasets.MNIST(root=ROOT, train=True,  transform=transform, download=True)
    test_set  = datasets.MNIST(root=ROOT, train=False, transform=transform, download=True)

    def stratified_subset(dataset, samples_per_class, num_classes=10):
        # robust ambil label (pakai .targets kalau ada)
        if hasattr(dataset, "targets"):
            labels = np.array(dataset.targets)
        else:
            labels = np.array([dataset[i][1] for i in range(len(dataset))])

        selected_idx = []
        rng = np.random.default_rng()  # biar cepat & reproducible kalau set seed di atas
        for c in range(num_classes):
            idx_c = np.where(labels == c)[0]
            chosen = rng.choice(idx_c, size=samples_per_class, replace=False)
            selected_idx.extend(chosen.tolist())
        return torch.utils.data.Subset(dataset, selected_idx)

    train_subset = stratified_subset(train_set, 1200, num_classes=10)
    test_subset  = stratified_subset(test_set,  200,  num_classes=10)

    def one_hot_encode(labels, num_classes=10):
        return torch.eye(num_classes, device=device)[labels]

    def collate_fn(batch):
        imgs, lbls = zip(*batch)
        imgs = torch.stack(imgs).to(device)         # (B,1,22,22)
        patches = extract_patches(imgs)             # (B,100,9)
        lbls = torch.tensor(lbls, dtype=torch.long, device=device)
        return patches, one_hot_encode(lbls, num_classes=10)

    train_loader = torch.utils.data.DataLoader(
        train_subset, batch_size=64, shuffle=True,  collate_fn=collate_fn
    )
    test_loader  = torch.utils.data.DataLoader(
        test_subset,  batch_size=64, shuffle=False, collate_fn=collate_fn
    )
    print(f"[MNIST] train_subset: {len(train_subset)}, test_subset: {len(test_subset)}")
    return train_loader, test_loader

# panggil loader
train_loader, test_loader = load_mnist()

In [ ]:
# ===== CELL 3 : QIFE Device & Model =====
# QIFE
n_qubits_qfe1      = 9
ansatz_layers_qfe1 = 1
# patch per gambar
n_patches = 100

# QAOA-Heuristic Ansatz dengan parameter berbeda untuk RZ dan RX
# No-Reupload Ansatz: QAOA-inspired tanpa reupload input per layer
def no_reupload_ansatz(inputs, params_rz, params_rx, num_layers, qubit_offset=0):
    # Embed input SEKALI di awal (tidak diulang per layer)
    qml.AngleEmbedding(inputs, wires=range(qubit_offset, qubit_offset + n_qubits_qfe1), rotation="Y")
    qml.AngleEmbedding(inputs, wires=range(qubit_offset, qubit_offset + n_qubits_qfe1), rotation="X")

    # Lapis parameter + entangler seperti sebelumnya
    for layer in range(num_layers):
        # Hadamard
        for i in range(qubit_offset, qubit_offset + n_qubits_qfe1):
            qml.Hadamard(wires=i)
        # Ring-CNOT + RZ (disandwich oleh CNOT)
        for i in range(qubit_offset, qubit_offset + n_qubits_qfe1):
            j = qubit_offset + ((i - qubit_offset + 1) % n_qubits_qfe1)
            qml.CNOT(wires=[i, j])
            qml.RZ(params_rz[layer, i - qubit_offset], wires=j)
            qml.CNOT(wires=[i, j])
        # RX per qubit
        for i in range(qubit_offset, qubit_offset + n_qubits_qfe1):
            qml.RX(params_rx[layer, i - qubit_offset], wires=i)

# Observables QIFE: ring ZZ (9 obs)
pairs_qfe1 = [(i, (i + 1) % n_qubits_qfe1) for i in range(n_qubits_qfe1)]
obs_list_qfe1 = [(i, j, 'Z','Z') for (i, j) in pairs_qfe1]

# Device & QNode
dev1 = qml.device("default.qubit", wires=n_qubits_qfe1)

@qml.qnode(dev1, interface="torch", diff_method='best')
def qnode_qfe1(inputs, params_rz, params_rx):
    inputs    = torch.as_tensor(inputs,    dtype=torch.float32)
    params_rz = torch.as_tensor(params_rz, dtype=torch.float32)
    params_rx = torch.as_tensor(params_rx, dtype=torch.float32)

    # PAKAI no-reupload (embed sekali di awal)
    no_reupload_ansatz(
        inputs,
        params_rz.view(ansatz_layers_qfe1, n_qubits_qfe1),
        params_rx.view(ansatz_layers_qfe1, n_qubits_qfe1),
        num_layers=ansatz_layers_qfe1
    )

    return [qml.expval(getattr(qml, P)(i) @ getattr(qml, Q)(j)) for (i, j, P, Q) in obs_list_qfe1]

class QFE1Extractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.q_layer = qml.qnn.TorchLayer(
            qnode_qfe1,
            weight_shapes={"params_rz": (ansatz_layers_qfe1, n_qubits_qfe1),
                           "params_rx": (ansatz_layers_qfe1, n_qubits_qfe1)}
        )
        # project 9*100 = 900 → 18
        self.project = nn.Linear(len(obs_list_qfe1) * n_patches, 18).to(device)

    def forward(self, x):
        b, n_p, p = x.shape          # (batch, 100, 9)
        flat = x.view(b * n_p, p)    # (b*100, 9)
        feats = self.q_layer(flat)   # (b*100, 9)
        feats = feats.view(b, n_p * feats.shape[1])  # (b, 900)
        return torch.tanh(self.project(feats)) * np.pi  # (b, 18)

class QuantumCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.qfe1  = QFE1Extractor()           # → (batch, 18)

        # fc1 terima 18 dim
        self.fc1  = nn.Linear(18, 128).to(device)
        self.fc2  = nn.Linear(128, 64).to(device)
        self.fc3  = nn.Linear(64, 10).to(device)

        for layer in (self.fc1, self.fc2):
            nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
            nn.init.zeros_(layer.bias)

    def forward(self, x):
        proj_feats = self.qfe1(x)             # (batch, 18)
        z = torch.relu(self.fc1(proj_feats))
        z = torch.relu(self.fc2(z))
        return self.fc3(z)

# Visualizers
def draw_quantum_circuit_qfe1():
    w_rz = np.random.randn(ansatz_layers_qfe1, n_qubits_qfe1)
    w_rx = np.random.randn(ansatz_layers_qfe1, n_qubits_qfe1)
    inputs = np.random.rand(n_qubits_qfe1)
    fig, ax = qml.draw_mpl(qnode_qfe1)(inputs, w_rz, w_rx)
    plt.title("Quantum Inspired Feature Extraction (QIFE)")
    plt.show()

draw_quantum_circuit_qfe1()

model = QuantumCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0005)
loss_fn   = nn.CrossEntropyLoss()

# Tampilkan parameter count
num_params = sum(p.numel() for p in model.parameters())
num_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Parameters: {num_params}")
print(f"Trainable Parameters: {num_trainable}")

In [ ]:
# ===== CELL 4 : Training berbasis Iteration =====
import time

# Fungsi evaluasi
# REPLACE fungsi evaluate() (versi multiclass macro-OVR):
def evaluate(model, data_loader):
    loss_sum, correct, total = 0.0, 0, 0
    y_true, y_pred = [], []
    y_proba_rows = []
    model.eval()
    with torch.no_grad():
        for images, labels in data_loader:
            outputs = model(images)  # (B,10)
            loss = loss_fn(outputs, labels.argmax(dim=1))
            loss_sum += loss.item()

            preds = outputs.argmax(dim=1).cpu().numpy()
            trues = labels.argmax(dim=1).cpu().numpy()
            probs = torch.softmax(outputs, dim=1).cpu().numpy()  # (B,10)

            y_pred.extend(preds); y_true.extend(trues)
            y_proba_rows.append(probs)

            correct += (preds == trues).sum()
            total   += len(trues)

    acc  = correct / total
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec  = recall_score   (y_true, y_pred, average='macro', zero_division=0)
    f1   = f1_score       (y_true, y_pred, average='macro', zero_division=0)
    try:
        y_proba = np.vstack(y_proba_rows)  # (N,10)
        auc = roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro')
    except ValueError:
        auc = float('nan')
    return loss_sum / len(data_loader), acc, prec, rec, f1, auc, y_true, y_pred

# Loop training berbasis iteration dengan progress bar
def train_iterations(model,
                     train_loader,
                     test_loader,
                     max_iterations=100,
                     eval_interval=100):

    start_time = time.perf_counter()   # ← pakai perf_counter
    fc_time_acc = 0.0
    train_losses, train_accs = [], []
    test_losses, test_accs = [], []
    test_precs, test_recs, test_f1s, test_aucs = [], [], [], []

    loader_iter = iter(train_loader)
    pbar = tqdm(total=max_iterations,
                desc="Iter 0/{}".format(max_iterations),
                ncols=100, leave=True)

    for iteration in range(1, max_iterations + 1):
        pbar.set_description(f"Iter {iteration}/{max_iterations}")

        try:
            images, labels = next(loader_iter)
        except StopIteration:
            loader_iter = iter(train_loader)
            images, labels = next(loader_iter)

        model.train()
        optimizer.zero_grad()

        # 1) QFE1 (timing tetap di-comment sesuai kode asli)
        proj_feats = model.qfe1(images)            # → (batch, 18)

        # 2) (tanpa attention) gunakan langsung fitur QFE1
        feats_in = proj_feats                      # (batch, 18)

        # 3) FC timing
        feats = feats_in.to(device, dtype=torch.float32).contiguous()
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        t0 = time.perf_counter()
        z  = torch.relu(model.fc1(feats))
        z  = torch.relu(model.fc2(z))
        outputs = model.fc3(z)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        fc_time_acc += (t1 - t0)

        loss = loss_fn(outputs, labels.argmax(dim=1))
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        batch_acc = (outputs.argmax(1) == labels.argmax(1)).float().mean().item()
        train_accs.append(batch_acc)

        pbar.set_postfix({"Loss": f"{loss.item():.4f}",
                          "Acc":  f"{batch_acc:.4f}"})
        pbar.update(1)

        if iteration % eval_interval == 0 or iteration == max_iterations:
            (test_loss, test_acc, test_prec, test_rec,
             test_f1, test_auc, y_true, y_pred) = evaluate(model, test_loader)

            test_losses.append(test_loss)
            test_accs.append(test_acc)
            test_precs.append(test_prec)
            test_recs.append(test_rec)
            test_f1s.append(test_f1)
            test_aucs.append(test_auc)

            tqdm.write(
                f"→ Eval @Iter {iteration}: "
                f"Test Loss={test_loss:.4f}, Test Acc={test_acc:.4f}, "
                f"Prec={test_prec:.4f}, Rec={test_rec:.4f}, F1={test_f1:.4f}, "
                f"AUC={test_auc:.4f}"
            )

            # Confusion matrix (label numerik 0..K-1)
            cm = confusion_matrix(y_true, y_pred)
            disp = ConfusionMatrixDisplay(cm, display_labels=list(range(cm.shape[0])))
            fig, ax = plt.subplots(figsize=(6,6))
            disp.plot(ax=ax, cmap='Blues', colorbar=False)
            ax.set_title(f"QIFE - MNIST")
            plt.show()

    pbar.close()
    total_time = time.perf_counter() - start_time   # ← pakai perf_counter
    print(f"\nTotal training time: {total_time:.4f}s")
    print(f"Total FC-Network computation time: {fc_time_acc:.4f}s")

    # Sumbu untuk grafik
    x_iter = list(range(1, max_iterations + 1))
    x_test = list(range(eval_interval, max_iterations+1, eval_interval))
    if x_test and x_test[-1] != max_iterations:
        x_test.append(max_iterations)

    # Grafik 1: Train Loss vs Iteration
    plt.figure(figsize=(6,4))
    plt.plot(x_iter, train_losses, label='Train Loss')
    plt.plot(x_test, test_losses, label='Test Loss')
    plt.xlabel('Iteration')
    plt.ylabel('Loss')
    plt.ylim(0, 3)
    plt.legend()
    plt.title('Loss vs Iteration')
    plt.show()

    # Grafik 2: Train & Test Accuracy vs Iteration
    plt.figure(figsize=(6,4))
    plt.plot(x_iter, train_accs, label='Train Acc')
    plt.plot(x_test, test_accs, label='Test Acc')
    plt.xlabel('Iteration')
    plt.ylabel('Accuracy')
    plt.ylim(0, 1)
    plt.legend()
    plt.title('Accuracy vs Iteration')
    plt.show()

    # Grafik 3: gabungan TRAIN (loss & acc)
    train_loss_color = 'tab:orange'
    train_acc_color  = 'tab:green'
    fig, ax1 = plt.subplots(figsize=(6,4))
    ax2 = ax1.twinx()
    ax1.plot(x_iter, train_losses, color=train_loss_color, label='Batch Loss')
    ax2.plot(x_iter, train_accs,  color=train_acc_color,  label='Batch Acc')
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Loss')
    ax2.set_ylabel('Accuracy')
    ax1.set_ylim(0, 3)
    ax2.set_ylim(0, 1)
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')
    plt.title('Train Performance')
    plt.show()

    # Grafik 4: gabungan TEST (loss & acc)
    test_loss_color = 'tab:orange'
    test_acc_color  = 'tab:green'
    fig, ax1 = plt.subplots(figsize=(6,4))
    ax2 = ax1.twinx()
    ax1.plot(x_test, test_losses, color=test_loss_color, label='Test Loss')
    ax2.plot(x_test, test_accs,  color=test_acc_color,  label='Test Acc')
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Loss')
    ax2.set_ylabel('Accuracy')
    ax1.set_ylim(0, 3)
    ax2.set_ylim(0, 1)
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')
    plt.title('Test Performance')
    plt.show()

# Panggil training
train_iterations(model, train_loader, test_loader,
                 max_iterations=100,
                 eval_interval=100)

In [ ]:
# ===== CELL 5 : Menyimpan bobot model yang sudah terlatih =====
def save_model(model, filename="QIFE_MNIST_iFran.pth"):
    torch.save(model.state_dict(), filename)
    print(f"Model state_dict saved to {filename}")

save_model(model)